In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Intrările și ieșirile Estimator

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Această pagină oferă o prezentare generală a intrărilor și ieșirilor primitivei Qiskit Runtime Estimator, care execută volumele de lucru pe resursele de calcul IBM Quantum&reg;. Estimator îți permite să definești eficient volume de lucru vectorizate folosind o structură de date numită [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). Acestea sunt folosite ca intrări pentru metoda [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) a primitivei Estimator, care execută volumul de lucru definit ca un job. Apoi, după ce jobul s-a finalizat, rezultatele sunt returnate într-un format care depinde atât de PUB-urile utilizate, cât și de opțiunile de runtime specificate pentru primitivă.
## Intrări
Fiecare PUB are acest format:

(`<un singur circuit>`, `<unul sau mai mulți observabili>`, `<opțional una sau mai multe valori de parametri>`, `<precizie opțională>`),

Opționalele `valori de parametri` pot fi o listă sau un singur parametru. Elementele din observabili și valorile de parametri sunt combinate urmând regulile de broadcasting NumPy descrise în subiectul [Intrările și ieșirile primitivelor](primitive-input-output#broadcasting-rules), iar o estimare a valorii de așteptare este returnată pentru fiecare element al formei broadcast.

> **Note:** Dacă intrarea conține măsurători, acestea sunt ignorate.

Pentru primitiva Estimator, un PUB poate conține cel mult patru valori:
- Un singur `QuantumCircuit`, care poate conține unul sau mai multe obiecte [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- O listă cu unul sau mai mulți observabili, care specifică valorile de așteptare ce urmează a fi estimate, aranjate într-un tablou (de exemplu, un singur observabil reprezentat ca un tablou 0-d, o listă de observabili ca un tablou 1-d și altele). Datele pot fi în oricare dintre formatele `ObservablesArrayLike`, cum ar fi `Pauli`, `SparsePauliOp`, `PauliList` sau `str`.
   > **Note:** - Observabilii comutanți **în același PUB** sunt grupați împreună folosind [această metodă](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Observabilii comutanți în PUB-uri diferite, chiar dacă au același circuit, nu sunt estimați folosind aceeași măsurătoare. Fiecare PUB reprezintă o bază diferită de măsurare și, prin urmare, sunt necesare măsurători separate pentru fiecare PUB.
>    - Pentru a te asigura că observabilii comutanți sunt estimați folosind aceeași măsurătoare, grupează-i în același PUB.
- O colecție de valori de parametri pentru a lega circuitul. Aceasta poate fi specificată ca un singur obiect de tip tablou unde ultimul index este pentru obiectele `Parameter` ale circuitului sau omisă (sau echivalent, setată la `None`) dacă circuitul nu are obiecte `Parameter`.
- (Opțional) O precizie țintă pentru valorile de așteptare ce urmează a fi estimate
---

Codul următor demonstrează un exemplu de set vectorizat de intrări pentru primitiva `Estimator` și le execută pe un backend IBM&reg; ca un singur obiect `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Ieșiri
După ce unul sau mai multe PUB-uri sunt trimise unui QPU pentru execuție și un job se finalizează cu succes, datele sunt returnate ca un obiect container [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) accesat prin apelarea metodei `RuntimeJobV2.result()`.

`PrimitiveResult` conține o listă iterabilă de obiecte [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) care conțin rezultatele execuției pentru fiecare PUB.

Fiecare element al acestei liste corespunde fiecărui PUB trimis metodei `run()` a primitivei (de exemplu, un job trimis cu 20 de PUB-uri va returna un obiect `PrimitiveResult` care conține o listă de 20 de obiecte `PubResult`, câte unul corespunzând fiecărui PUB).

Fiecare `PubResult` pentru primitiva Estimator conține cel puțin un tablou de valori de așteptare (`PubResult.data.evs`) și deviații standard asociate (fie `PubResult.data.stds`, fie `PubResult.data.ensemble_standard_error` în funcție de `resilience_level` utilizat), dar poate conține mai multe date în funcție de opțiunile de atenuare a erorilor specificate.

Fiecare obiect `PubResult` posedă atât un atribut `data`, cât și un atribut `metadata`.
    - Atributul `data` este un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizat care conține valorile efective ale măsurătorilor, deviațiile standard și altele.
    - `DataBin` are diverse atribute în funcție de forma sau structura PUB-ului asociat, precum și de opțiunile de atenuare a erorilor specificate de primitiva folosită pentru a trimite jobul (de exemplu, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) sau [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Atributul `metadata` conține informații despre opțiunile de runtime și de atenuare a erorilor utilizate (explicate ulterior în secțiunea [Metadatele rezultatelor](#result-metadata) de pe această pagină).

Următoarea este o schiță vizuală a structurii de date `PrimitiveResult` pentru ieșirea Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Pe scurt, un singur job returnează un obiect `PrimitiveResult` și conține o listă cu unul sau mai multe obiecte `PubResult`. Aceste obiecte `PubResult` stochează apoi datele de măsurare pentru fiecare PUB trimis jobului.

Fragmentul de cod de mai jos descrie formatul `PrimitiveResult` (și `PubResult` asociat) pentru jobul creat mai sus.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
